In [13]:
%pip install numpy datasets google.generativeai python_dotenv matplotlib seaborn scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 36.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Cell 1: Fixed Dataset Loading
import numpy as np
from datasets import load_dataset

def load_and_sample_stories():
    dataset = load_dataset('euclaise/writingprompts', split='train')
    indices = np.random.choice(len(dataset), 120, replace=False)
    indices = [int(i) for i in indices]  # Convert numpy.int64 to native int
    return [dataset[i]['story'] for i in indices], indices

stories, story_indices = load_and_sample_stories()


In [ ]:
# Cell 2: Sequential API Calls with Rate Limiting and Incremental Saving
import google.generativeai as genai
import os
import re
import time
from dotenv import load_dotenv
from datetime import datetime
import json

load_dotenv()
genai.configure(api_key=os.getenv('GEMINI_API_KEY'))

def process_story_with_gemini_sequential(story, index):
    model = genai.GenerativeModel('gemini-1.5-flash')
    
    # Call 1: Objective summary
    prompt_summary = f"""Provide an objective summary of the following story. Refer to the narrator as <the narrator>.

Story: {story[:20000]}"""
    response_summary = model.generate_content(prompt_summary)
    summary = response_summary.text.strip()
    print(f"  - Summary completed")
    
    # Call 2: List of characters
    prompt_characters = f"""List all characters in the following story. Include all named entities.

Story: {story[:20000]}"""
    response_characters = model.generate_content(prompt_characters)
    characters = [c.strip() for c in response_characters.text.strip().split('\n') if c.strip()]
    print(f"  - Characters completed")
    
    # Call 3: Short camelCase story ID
    prompt_name = f"""Create a short camelCase name for the following story. This will be used as a story ID.

Story: {story[:20000]}"""
    response_name = model.generate_content(prompt_name)
    name = response_name.text.strip()
    print(f"  - Name completed")
    
    # Call 4: List of environments/settings
    prompt_environment = f"""List all environments and settings in the following story.

Story: {story[:20000]}"""
    response_environment = model.generate_content(prompt_environment)
    environment = [e.strip() for e in response_environment.text.strip().split('\n') if e.strip()]
    print(f"  - Environment completed")
    
    return {
        "storyNum": int(index),
        "name": name,
        "source": story,
        "summary": summary,
        "characters": characters,
        "environment": environment
    }

def save_current_progress(data, filename):
    """Save current progress to file"""
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"Progress saved to {filename}")

# Initialize filename for incremental saving
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f"extracted_{timestamp}.json"

# Process stories and save results incrementally
extracted_data = []
for idx, (story, story_idx) in enumerate(zip(stories, story_indices)):
    time.sleep(10)  # Rate limiting
    print(f"Processing story {idx+1}/120 (index: {story_idx})")
    try:
        result = process_story_with_gemini_sequential(story, story_idx)
        extracted_data.append(result)
        # Save after each successful processing
        save_current_progress(extracted_data, filename)
        print(f"Completed {idx+1}/120 stories")
    except Exception as e:
        print(f"Error processing story {idx+1} (index: {story_idx}): {str(e)}")
        # Save even if there's an error
        save_current_progress(extracted_data, filename)
        print("Continuing with next story...")
    
    # Additional rate limiting between stories
    if idx < len(stories) - 1:  # Don't sleep after the last story
        print("Waiting 5 seconds before next story...")
        time.sleep(5)

print(f"All processing complete. Final data saved to {filename}")


In [ ]:
# Cell 4: Inference Functions with Ollama/Mistral Implementation
import csv
import time
import os
import requests
import json

def generate_retelling(extracted_data, output_dir="output"):
    # Format the prompt with data from the extracted story
    characters_str = ", ".join(extracted_data["characters"])
    environments_str = ", ".join(extracted_data["environment"])
    
    prompt_template = f"""Here is a list of objective, third person facts from a story:
    Characters: {characters_str}
    Settings: {environments_str}
    Original story: {extracted_data["summary"]}
    
    Using the information provided, craft a compelling retelling of the story that is from a perspective other than the narrator's. Choose one of the characters or elements in the story to narrate from their point of view."""
    
    # Call Ollama API running locally with Mistral
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "mistral:latest",
                "prompt": prompt_template,
                "stream": False
            },
            timeout=120  # 2-minute timeout
        )
        
        # Check if request was successful
        if response.status_code == 200:
            result = response.json()
            retelling = result.get("response", "Error: No response generated")
        else:
            retelling = f"Error: Received status code {response.status_code} from Ollama API"
            print(f"API Error: {response.text}")
    
    except requests.exceptions.RequestException as e:
        retelling = f"Error connecting to Ollama: {str(e)}"
        print(f"Connection error: {str(e)}")
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate unique filename with timestamp
    timestamp = time.strftime('%m-%d-%H-%M-%S')
    filename = f"{output_dir}/retell_{extracted_data['name']}_{timestamp}.csv"
    
    # Save the retelling to CSV
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=["storyNum", "name", "source", "retelling"])
        writer.writeheader()
        writer.writerow({
            "storyNum": extracted_data["storyNum"],
            "name": extracted_data["name"],
            "source": extracted_data["source"],
            "retelling": retelling
        })
    
    print(f"Retelling saved to {filename}")
    return filename

def batch_generate_retellings(extracted_data_file:str, output_dir="output"):
    """Process all stories from an extracted data file"""
    
    # Load the extracted data
    with open(extracted_data_file, 'r') as f:
        extracted_data_list = json.load(f)
    
    # Process each story
    for idx, story_data in enumerate(extracted_data_list):
        print(f"Generating retelling {idx+1}/{len(extracted_data_list)} for '{story_data['name']}'")
        
        try:
            filename = generate_retelling(story_data, output_dir)
            print(f"Successfully generated retelling: {filename}")
        except Exception as e:
            print(f"Error generating retelling for story {story_data['name']}: {str(e)}")
    
    print(f"All retellings complete. Results saved to {output_dir}/")


In [ ]:
batch_generate_retellings("./extracted_20250504_032008.json")

In [ ]:
# Cell 5: Structured Evaluation Functions
import requests
import json
import os
import glob
import pandas as pd
import numpy as np

def evaluate_retelling(original, retelling):
    structured_prompt = f"""Evaluate this story retelling using these EXACT criteria:

Character Selection Valence (1-5):
1: Character selected is a secondary character within the story, or a character with a prominent role in the story that isn't the narrator
5: Character selected is highly original or unique; examples include choosing inanimate objects, or passing characters that have very little revealed about them in the original story


Character Selection Error (bool):
True if character not in original story

Factual Consistency (1-5):
1: The story is a one-for-one retelling of the events that occurred from the perspective of something or someone else, with very little narrative content added
5: The retelling contains nuanced perspectives that may have not been explored or talk about extensively in the original story's first person view

Factual Consistency Error (bool):
True if the retelling introduces a major factual discrepancy that changes the plot, genre, or character of the original story

Retelling Depth (1-5):
1: The perspective is almost entirely unrelated to the character; without knowledge of the prompt or source story, the retelling could belong to any other character in the story
5: 5: For the chosen character, the depicted perspective is nuanced and original, and demonstrates a thorough understanding of their intentions, goals, and personalities 

Original Story: {original[:5000]}
Retelling: {retelling[:5000]}

Respond ONLY with JSON containing these keys:
- valence (int 1-5)
- selection_error (bool)
- consistency (int 1-5)
- fact_error (bool)
- retelling_depth (int 1-5)
- reasoning (object with keys: valence, consistency, retelling_depth)"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "llama3:8b",
                "prompt": structured_prompt,
                "format": "json",
                "stream": False
            },
            timeout=300
        )
        
        if response.status_code == 200:
            try:
                result = json.loads(response.json()["response"])
                # Convert numpy types to native Python types
                return {
                    "valence": int(result["valence"]),
                    "selection_error": bool(result["selection_error"]),
                    "consistency": int(result["consistency"]),
                    "fact_error": bool(result["fact_error"]),
                    "retelling_depth": int(result["retelling_depth"]),
                    "reasoning": {
                        "valence": str(result["reasoning"]["valence"]),
                        "consistency": str(result["reasoning"]["consistency"]),
                        "retelling_depth": str(result["reasoning"]["retelling_depth"])
                    }
                }
            except (KeyError, json.JSONDecodeError) as e:
                return {"error": f"Invalid response format: {str(e)}"}
        else:
            return {"error": f"API error: {response.status_code}"}
    
    except Exception as e:
        return {"error": str(e)}

def batch_evaluate(output_dir):
    timestamp = pd.Timestamp.now().strftime("%m-%d-%H-%M-%S")
    eval_file = os.path.join(output_dir, f"eval_results_{timestamp}.jsonl")
    
    for file in glob.glob(os.path.join(output_dir, "retell_*.csv")):
        try:
            # Read CSV with dtype specification
            df = pd.read_csv(file, dtype={
                'storyNum': 'int32',
                'name': 'str',
                'source': 'str',
                'retelling': 'str'
            })
            
            # Convert pandas/numpy types to native Python types
            record = {
                "storyNum": int(df["storyNum"].iloc[0].item()),  # Convert int64 to int
                "name": str(df["name"].iloc[0]),
                "source": str(df["source"].iloc[0]),
                "retelling": str(df["retelling"].iloc[0])
            }
            
            print(f"Evaluating {record['name']}...")
            evaluation = evaluate_retelling(record["source"], record["retelling"])
            
            # Merge results with explicit type conversion
            combined = {
                **record,
                **{k: v.item() if isinstance(v, np.generic) else v 
                   for k, v in evaluation.items()}
            }
            
            # Save immediately with UTF-8 encoding
            with open(eval_file, 'a', encoding='utf-8') as f:
                json.dump(combined, f, ensure_ascii=False)
                f.write('\n')
                
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
            error_entry = {
                "file": file,
                "error": str(e),
                "timestamp": pd.Timestamp.now().isoformat()
            }
            with open(eval_file, 'a', encoding='utf-8') as f:
                json.dump(error_entry, f, ensure_ascii=False)
                f.write('\n')

    print(f"Evaluation complete. Results saved to {eval_file}")
    return eval_file


In [15]:
batch_evaluate("./output")

Evaluating DesertRevenge...
Evaluating GodsBetrayalAndRedemption...
Evaluating HarlemHold...
Evaluating DevinsCursedCoins...
Evaluating FlorentineEscape...
Evaluating AmnesiacCallCenterDevil...
Evaluating ThirtyYearsThirtyLives...
Evaluating RedHairedNord...
Evaluating FinalBirthTunnel...
Evaluating ericMayaGhost...
Evaluating NebulonTerrierIncident...
Evaluating gelliusBathDeath...
Evaluating ShipwreckedGenieWish...
Evaluating woodsBodyMystery...
Evaluating CokeEscape...
Evaluating BlackHatWhiteHatRebirth...
Evaluating LostLifetimeJourney...
Evaluating HiddenDifference...
Evaluating FallenSoldier...
Evaluating OrphanageAnomaly...
Evaluating SuicideHotlineCall...
Evaluating OceanSafeRepository...
Evaluating SuddenDarkness...
Evaluating JerrysVirtualVacation...
Evaluating GameChildrensDiscovery...
Evaluating EarthCommandPrompt...
Evaluating SilentFogEncounter...
Evaluating LicenseAndRegistrationPlease...
Evaluating ForgottenCelebration...
Evaluating GoneBabyBoy...
Evaluating GrandpaRedD

'./output/eval_results_05-05-03-25-12.jsonl'

In [7]:
import os
import glob
import pandas as pd

# Get the directory of this notebook
NOTEBOOK_DIR = os.getcwd()

# Paths configuration
FINAL_PROJECT_DIR = NOTEBOOK_DIR
EXTRACTED_PATTERN = os.path.join(FINAL_PROJECT_DIR, 'extracted_*.json')
EVAL_PATTERN = os.path.join(FINAL_PROJECT_DIR, 'eval_results_*.jsonl')

def load_data():
    # Verify directory structure
    if not os.path.exists(FINAL_PROJECT_DIR):
        raise FileNotFoundError(f"""
        'final project' directory not found at: {FINAL_PROJECT_DIR}
        Current directory contents: {os.listdir(NOTEBOOK_DIR)}
        """)
    
    # Find latest extracted file
    extracted_files = glob.glob(EXTRACTED_PATTERN)
    if not extracted_files:
        raise FileNotFoundError(f"No extracted files found matching: {EXTRACTED_PATTERN}")
    
    # Find latest evaluation file
    eval_files = glob.glob(EVAL_PATTERN)
    if not eval_files:
        raise FileNotFoundError(f"No evaluation files found matching: {EVAL_PATTERN}")
    
    # Load data
    extracted_df = pd.read_json(max(extracted_files, key=os.path.getctime))
    eval_df = pd.concat([pd.read_json(f, lines=True) for f in eval_files])
    
    # Merge and analyze
    merged_df = pd.merge(
        eval_df,
        extracted_df[["storyNum", "characters", "environment"]],
        on="storyNum",
        how="inner"
    ).assign(
        num_characters = lambda df: df['characters'].apply(len),
        num_settings = lambda df: df['environment'].apply(len)
    )
    
    return merged_df

# Usage
try:
    df = load_data()
    print("Data loaded successfully!\n")
    print("Columns:", df.columns.tolist())
    print("\nSample data:")
    print(df[['storyNum', 'valence', 'retelling_depth', 'num_characters']].head())
    print("\nSummary stats:")
    print(df[['valence', 'retelling_depth', 'num_characters', 'num_settings']].describe())
    
except FileNotFoundError as e:
    print(f"Error: {str(e)}")
    print("\nTroubleshooting steps:")
    print(f"1. Ensure 'final project' directory exists in: {NOTEBOOK_DIR}")
    print(f"2. Check for extracted JSON files in: {FINAL_PROJECT_DIR}")
    print(f"3. Verify evaluation JSONL files exist in: {FINAL_PROJECT_DIR}")
    print(f"\nCurrent 'final project' contents: {os.listdir(FINAL_PROJECT_DIR) if os.path.exists(FINAL_PROJECT_DIR) else 'Directory not found'}")


Data loaded successfully!

Columns: ['storyNum', 'name', 'source', 'retelling', 'valence', 'selection_error', 'consistency', 'fact_error', 'retelling_depth', 'reasoning', 'error', 'file', 'timestamp', 'characters', 'environment', 'num_characters', 'num_settings']

Sample data:
   storyNum  valence  retelling_depth  num_characters
0  149092.0      2.0              4.0              13
1  204554.0      4.0              4.0               8
2  265073.0      4.0              3.0               7
3  237430.0      3.0              4.0               6
4  138154.0      2.0              3.0               2

Summary stats:
          valence  retelling_depth  num_characters  num_settings
count  114.000000       114.000000      119.000000    119.000000
mean     3.131579         4.078947        6.084034      7.932773
std      1.034905         0.765919        3.373827      3.948248
min      2.000000         2.000000        1.000000      1.000000
25%      2.000000         4.000000        4.000000      6

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, linregress

def analyze_evaluation_data(data_dir):
    # 1. Create analysis directory
    analysis_dir = os.path.join(data_dir, 'analysis')
    os.makedirs(analysis_dir, exist_ok=True)
    
    # 2. File discovery and loading
    extracted_files = glob.glob(os.path.join(data_dir, 'extracted_*.json'))
    eval_files = glob.glob(os.path.join(data_dir, 'eval_results_*.jsonl'))
    if not extracted_files or not eval_files:
        print("Error: Missing data files")
        print(f"Extracted files found: {len(extracted_files)}")
        print(f"Evaluation files found: {len(eval_files)}")
        return

    # 3. Load data
    extracted_df = pd.read_json(max(extracted_files, key=os.path.getctime))
    eval_df = pd.concat([pd.read_json(f, lines=True) for f in eval_files])
    merged_df = pd.merge(
        eval_df,
        extracted_df[["storyNum", "characters", "environment"]],
        on="storyNum",
        how="inner"
    )
    original_count = len(merged_df)
    merged_df["num_characters"] = merged_df["characters"].apply(len)
    merged_df["num_settings"] = merged_df["environment"].apply(len)
    missing_values = merged_df[['num_characters', 'num_settings', 'valence', 'consistency', 'retelling_depth']].isna().sum()

    # 4. Remove constant/unhashable columns
    constant_cols = []
    unhashable_cols = []
    for col in merged_df.columns:
        try:
            if merged_df[col].nunique() == 1:
                constant_cols.append(col)
        except TypeError:
            unhashable_cols.append(col)
    clean_df = merged_df.drop(columns=constant_cols + unhashable_cols)
    clean_df = clean_df.dropna(subset=['num_characters', 'num_settings', 'valence', 'consistency', 'retelling_depth'])
    clean_count = len(clean_df)

        # 5. Save diagnostics
    with open(os.path.join(analysis_dir, 'diagnostics.txt'), 'w') as f:
        f.write(f"Original records: {original_count}\n")
        f.write(f"Valid records after cleaning: {clean_count}\n")
        f.write(f"Dropped records: {original_count - clean_count}\n")
        f.write("\nMissing values (pre-cleaning):\n")
        f.write(missing_values.to_string() + "\n\n")
        f.write("Constant columns (excluded from analysis):\n")
        f.write(f"{constant_cols or 'None'}\n")
        f.write(f"Unhashable columns (excluded): {unhashable_cols or 'None'}\n")

    # New: Evaluation metrics summary statistics
    eval_metrics = ['valence', 'consistency', 'retelling_depth']
    summary_stats = clean_df[eval_metrics].describe().transpose()
    summary_stats = summary_stats[['count', 'mean', 'std', 'min', '50%', 'max']]
    summary_stats.rename(columns={'50%': 'median'}, inplace=True)

    with open(os.path.join(analysis_dir, 'evaluation_metrics_summary.txt'), 'w') as f:
        f.write("Evaluation Metrics Summary Statistics:\n\n")
        f.write(summary_stats.round(3).to_string())
        f.write("\n\nKey:\n")
        f.write("- count: Number of valid observations\n")
        f.write("- mean: Average score\n")
        f.write("- std: Standard deviation\n")
        f.write("- min: Minimum observed score\n")
        f.write("- median: Middle value\n")
        f.write("- max: Maximum observed score\n")

    # 6. Correlation and plotting
    results = []
    for dep in ["valence", "consistency", "retelling_depth"]:
        for indep in ["num_characters", "num_settings"]:
            if indep not in clean_df or dep not in clean_df:
                continue
            # Spearman correlation
            valid = ~np.isnan(clean_df[indep]) & ~np.isnan(clean_df[dep])
            x = clean_df.loc[valid, indep]
            y = clean_df.loc[valid, dep]
            if len(x) < 2:
                continue
            spearman_r, spearman_p = spearmanr(x, y)
            # Linear regression
            try:
                slope, intercept, r_value, p_value, std_err = linregress(x, y)
            except Exception:
                r_value, p_value = np.nan, np.nan
            results.append({
                'independent': indep,
                'dependent': dep,
                'spearman_r': spearman_r,
                'spearman_p': spearman_p,
                'linear_r2': r_value**2 if not np.isnan(r_value) else np.nan,
                'linear_p': p_value,
                'n': len(x)
            })
            # Plot
            sns.set_style("whitegrid", {
                "grid.color": (0.9, 0.9, 0.9),  # RGB tuple, values 0-1
            })

            plt.figure(figsize=(8, 6))
            sns.scatterplot(x=x, y=y, alpha=0.7)
            sns.regplot(x=x, y=y, scatter=False, color='red')
            text = (f"Spearman r = {spearman_r:.2f} (p = {spearman_p:.3f})\n"
                    f"Linear R² = {r_value**2:.2f} (p = {p_value:.3f})\n"
                    f"N = {len(x)}")
            plt.annotate(text, xy=(0.05, 0.95), xycoords='axes fraction',
                         ha='left', va='top', fontsize=10,
                         bbox=dict(boxstyle='round', fc='wheat', alpha=0.8))
            plt.xlabel(indep)
            plt.ylabel(dep)
            plt.title(f"{indep} vs {dep}")
            plt.tight_layout()
            plot_path = os.path.join(analysis_dir, f"{indep}_vs_{dep}.png")
            plt.savefig(plot_path)
            plt.close()

    # 7. Save correlation results
    results_df = pd.DataFrame(results)
    results_df.to_csv(os.path.join(analysis_dir, 'correlation_results.csv'), index=False)

    # 8. Save cleaned data
    clean_df.to_csv(os.path.join(analysis_dir, 'cleaned_data.csv'), index=False)

    # 9. Pairplot overview
    if len(clean_df) >= 10:
        g = sns.pairplot(clean_df[['num_characters', 'num_settings', 'valence', 'consistency', 'retelling_depth']])
        g.fig.suptitle("Data Relationships Overview", y=1.02)
        pairplot_path = os.path.join(analysis_dir, 'pairplot_overview.png')
        plt.savefig(pairplot_path)
        plt.close()

    print(f"\nAnalysis complete. Results saved to: {analysis_dir}")

# Example usage:
# analyze_evaluation_data('./final project')


In [49]:
analyze_evaluation_data(data_dir="./")


Analysis complete. Results saved to: ./analysis
